In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os

from pydantic import BaseModel

from doc_intelligence import PDFProcessor

In [3]:
import sys

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from utils import show_pdf_with_bboxes

PDF_URL = "https://example-files.online-convert.com/document/pdf/example.pdf"
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output")
os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
class License(BaseModel):
    license_name: str

In [5]:
mp_processor = PDFProcessor(provider="openai", model="gpt-4o-mini")

In [6]:
mp_response = mp_processor.extract(
    uri=PDF_URL,
    response_format=License,
    include_citations=True,
    extraction_mode="multi_pass",
)

2026-03-13 02:42:41.777 | INFO     | doc_intelligence.pdf.processor:_parse:74 - Document parsed successfully
2026-03-13 02:42:41.778 | DEBUG    | doc_intelligence.llm:generate_text:26 - OpenAILLM: generate_text: Generating text with model: gpt-4o-mini
2026-03-13 02:42:43.738 | DEBUG    | doc_intelligence.pdf.extractor:_run_multi_pass:153 - DigitalPDFExtractor: multi-pass: pass1 complete: license_name='Attribution-ShareAlike 3.0 Unported'
2026-03-13 02:42:43.739 | DEBUG    | doc_intelligence.llm:generate_text:26 - OpenAILLM: generate_text: Generating text with model: gpt-4o-mini
2026-03-13 02:42:45.243 | DEBUG    | doc_intelligence.pdf.extractor:_run_multi_pass:161 - DigitalPDFExtractor: multi-pass: pass2 page_map: {'license_name': [0]}
2026-03-13 02:42:45.244 | INFO     | doc_intelligence.pdf.formatter:format_document_for_llm:53 - Formatting 1 pages
2026-03-13 02:42:45.244 | DEBUG    | doc_intelligence.llm:generate_text:26 - OpenAILLM: generate_text: Generating text with model: gpt-4o-

In [7]:
mp_response

ExtractionResult(data=License(license_name='Attribution-ShareAlike 3.0 Unported'), metadata={'license_name': {'value': 'Attribution-ShareAlike 3.0 Unported', 'citations': [{'page': 0, 'bboxes': [{'x0': 0.20106913928643427, 'top': 0.8587326111744586, 'x1': 0.5648947389639185, 'bottom': 0.8718454960091222}]}]}})

In [8]:
# NOTE: Multi-pass internals (pass1_result, pass2_page_map) are now
# encapsulated inside the processor and not exposed as public attributes.
# mp_doc = cast(PDFDocument, mp_processor.document)
# mp_doc.pass1_result

In [9]:
# mp_doc.pass2_page_map

In [10]:
mp_response

ExtractionResult(data=License(license_name='Attribution-ShareAlike 3.0 Unported'), metadata={'license_name': {'value': 'Attribution-ShareAlike 3.0 Unported', 'citations': [{'page': 0, 'bboxes': [{'x0': 0.20106913928643427, 'top': 0.8587326111744586, 'x1': 0.5648947389639185, 'bottom': 0.8718454960091222}]}]}})

In [11]:
assert mp_response.metadata is not None
show_pdf_with_bboxes(
    PDF_URL,
    mp_response.metadata["license_name"]["citations"][0],
    out_file=os.path.join(OUT_DIR, "multi_pass_bbox.png"),
)

  -> saved to /Users/zeel/Public/ms/open_source/document_ai/notebooks/pdf/output/multi_pass_bbox.png
